## Все доступные признаки в датасете

На основе вашего описания и выгруженных данных, вот полный список всех признаков, которые у вас есть для Entity Resolution:

---

### 1. **Основные демографические признаки**

| Признак | Тип | Описание | Использование в ER |
|---------|-----|----------|-------------------|
| `first_name` | string | Анонимизированное имя (род сохранён) | Сравнение имён |
| `last_name` | string | Анонимизированная фамилия (род сохранён) | Сравнение фамилий |
| `sex` | string | Пол (male/female/unknown) | Совпадение пола |
| `birthday` | date | Анонимизированная дата рождения (1 января не анонимизированы) | Совпадение даты рождения |
| `email` | string | Анонимизированный email (домен сохранён) | Точное совпадение, домен |
| `phone` | string | Анонимизированный телефон (код страны + код оператора сохранены) | Точное совпадение |

---

### 2. **Временные признаки**

| Признак | Тип | Описание | Использование в ER |
|---------|-----|----------|-------------------|
| `created_at` | timestamp | Время события | Сортировка, временные интервалы между событиями |
| `hour` | int (производный) | Час события (0-23) | Паттерны активности |
| `day_of_week` | int (производный) | День недели (0-6) | Паттерны активности |

---

### 3. **Признаки из `non_processing_features`**

Это массив строк с техническими и контекстными признаками. Уникальные значения:

| Признак | Тип | Уникальные значения | Использование в ER |
|---------|-----|---------------------|-------------------|
| `device` | string | `pc`, `smartphone`, `tablet` | Совпадение устройства |
| `osfamily` | string | `android`, `ios`, `linux`, `os-x`, `windows` | Совпадение ОС |
| `browser` | string | `chrome`, `firefox`, `opera`, `safari`, `yandex` | Совпадение браузера |
| `geoname_id` | int | 500+ значений (ID города в GeoNames) | Совпадение города |
| `subdivision_1_iso_code` | string | Множество кодов (ROS, MOS, SP, 01, 02, ...) | Совпадение региона |
| `is_not_russia` | bool | `None` (всегда None, т.е. все из России) | Не используется |

---

### 4. **Признаки из `realtime_features`**

JSON-строка с признаками, рассчитанными в момент события. Уникальные значения:

| Признак | Тип | Уникальные значения | Использование в ER |
|---------|-----|---------------------|-------------------|
| `country` | string | 113 стран (AE, AL, AM, ... RU, US, ...) | Совпадение страны |
| `is_million` | bool | `True`, `False` | Признак города-миллионника |
| `tz_offset` | int | -9 до +12 (целые часы) | Часовой пояс |
| `geoname` | string | 500+ названий городов | Совпадение города |
| `geoid` | int | 500+ ID городов GeoNames | Совпадение города |
| `local_hour` | int | 0-23 | Временные паттерны |
| `day` | int | 0-6 (день недели) | Временные паттерны |
| `population` | int | 500 до 10M+ | Размер города |
| `visit_count` | string | '0', '1', ... '226' | Количество визитов |

---

### 5. **Признаки из `fs_features` (Feature Store)**

Массив строк вида `feature_name:value` или флаги `feature_name`. Содержит:

#### A. **Site-id признаки (содержат маскированный site-id)**

| Признак | Значение |
|---------|----------|
| `has_accept_30:XXXX` | Принял предложение за 30 дней |
| `has_accept_365:XXXX` | Принял предложение за 365 дней |
| `has_account:XXXX` | Есть аккаунт на сайте |
| `has_click_30:XXXX` | Кликнул за 30 дней |
| `has_click_365:XXXX` | Кликнул за 365 дней |
| `has_order_30:XXXX` | Сделал заказ за 30 дней |
| `has_order_365:XXXX` | Сделал заказ за 365 дней |
| `has_view_90:XXXX` | Просматривал за 90 дней |
| `source_site_30:XXXX` | Источник трафика за 30 дней |
| `source_site_365:XXXX` | Источник трафика за 365 дней |
| `visited_30:XXXX` | Посещал за 30 дней |
| `visited_365:XXXX` | Посещал за 365 дней |

#### B. **Флаги (без значения)**

| Признак | Описание |
|---------|----------|
| `is_gmail` | Использует Gmail |
| `is_man` | Мужской пол |
| `is_phone` | Номер телефона подтверждён |
| `is_woman` | Женский пол |
| `is_yandex` | Использует Яндекс.Почту |
| `was_phone_lead` | Был лид по телефону |

#### C. **Категориальные признаки (postman — email-коммуникации)**

| Признак | Описание |
|---------|----------|
| `mm_event_30`, `mm_event_90` | Маркетинговое событие |
| `postman_action_30`, `postman_action_90` | Действие (open, click, smtp_open) |
| `postman_campaign_30`, `postman_campaign_90` | ID кампании |
| `postman_response_30`, `postman_response_90` | Результат (ok, fail) |
| `postman_scenario_30`, `postman_scenario_90` | Сценарий рассылки |

---

### 6. **Мета-признаки**

| Признак | Тип | Описание |
|---------|-----|----------|
| `profile_id` | string | Уникальный идентификатор профиля |
| `entity_id` | string | Идентификатор реального человека (целевая переменная) |
| `entity_type` | string | `single_profile` или `multi_profile` |
| `is_multi_entity` | bool | Производный признак (True для multi_profile) |



In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from itertools import combinations
from collections import defaultdict

import torch
import torch.nn.functional as F
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import NearestNeighbors

from catboost import CatBoostClassifier

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero

In [2]:
PROJECT_ROOT = Path(os.path.abspath('..'))
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATA_DIR = PROJECT_ROOT / 'data'

In [3]:
df = pd.read_csv(ARTIFACTS_DIR / 'df.csv')

In [4]:
# 1. Таблица уникальных entity_id для разбиения
entity_df = (
    df.groupby('entity_id', as_index=False)
      .agg(
          row_count=('entity_id', 'size'),
          is_multi_entity=('is_multi_entity', 'first')
      )
)

# 2. Делим entity_id на train+val и test
train_val_entity_df, test_entity_df = train_test_split(
    entity_df,
    test_size=0.20,
    random_state=42,
    stratify=entity_df['is_multi_entity']
)

# 3. Делим train+val на train и val
train_entity_df, val_entity_df = train_test_split(
    train_val_entity_df,
    test_size=0.25,   # 0.25 от 80% = 20% от общего объема
    random_state=42,
    stratify=train_val_entity_df['is_multi_entity']
)

# 4. Множества entity_id по сплитам
train_entities = set(train_entity_df['entity_id'])
val_entities = set(val_entity_df['entity_id'])
test_entities = set(test_entity_df['entity_id'])

# 5. Проставляем split всем строкам исходного df
df = df.copy()
df['split'] = np.select(
    [
        df['entity_id'].isin(train_entities),
        df['entity_id'].isin(val_entities),
        df['entity_id'].isin(test_entities),
    ],
    [
        'train',
        'validation',
        'test'
    ],
    default='unknown'
)

# 6. Проверка
print(df['split'].value_counts(dropna=False))
print(df.groupby('split')['entity_id'].nunique())
print(df[df['split'] == 'unknown'].shape)

split
train         40711
validation    13746
test          13579
Name: count, dtype: int64
split
test          10674
train         32021
validation    10674
Name: entity_id, dtype: int64
(0, 60)


In [5]:
print(train_entities & val_entities)
print(train_entities & test_entities)
print(val_entities & test_entities)

set()
set()
set()


In [6]:
df[df['split'] == 'train']['entity_id'].nunique()

32021

# Подготовка train
Сначала работаем только с train-сплитом, чтобы не было утечки. Это важно, потому что embeddings не должны обучаться на структуре validation/test-графа

In [7]:
df_train = df[df['split'] == 'train'].copy()

# приведение к строкам для стабильности
for col in ['profile_id', 'entity_id', 'email', 'phone', 'email_domain',
            'device', 'city_name', 'region_code', 'browser', 'osfamily', 'sex']:
    if col in df_train.columns:
        df_train[col] = df_train[col].astype(str)
        df_train.loc[df_train[col].isin(['nan', 'None']), col] = np.nan

# Агрегация до profile_id

In [8]:
def to_set(series):
    vals = series.dropna().astype(str).unique().tolist()
    return set(vals)

def mode_or_nan(series):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return np.nan
    return s.mode().iat[0]

profile_df = (
    df_train.groupby('profile_id')
    .agg(
        entity_id=('entity_id', mode_or_nan),
        row_count=('profile_id', 'size'),

        email_set=('email', to_set),
        phone_set=('phone', to_set),
        domain_set=('email_domain', to_set),
        device_set=('device', to_set),
        city_set=('city_name', to_set),
        region_set=('region_code', to_set),
        browser_set=('browser', to_set),
        os_set=('osfamily', to_set),
        sex_set=('sex', to_set),

        local_hour_mean=('local_hour', 'mean'),
        visit_count_mean=('visit_count', 'mean'),
        city_population_mean=('city_population', 'mean'),

        fs_is_phone_mean=('fs_is_phone', 'mean'),
        fs_is_gmail_mean=('fs_is_gmail', 'mean'),
        fs_is_yandex_mean=('fs_is_yandex', 'mean'),
        fs_is_man_mean=('fs_is_man', 'mean'),
        fs_is_woman_mean=('fs_is_woman', 'mean'),
    )
    .reset_index()
)

profile_df['n_email'] = profile_df['email_set'].apply(len)
profile_df['n_phone'] = profile_df['phone_set'].apply(len)
profile_df['n_domain'] = profile_df['domain_set'].apply(len)
profile_df['n_device'] = profile_df['device_set'].apply(len)
profile_df['n_city'] = profile_df['city_set'].apply(len)
profile_df['n_region'] = profile_df['region_set'].apply(len)

In [9]:
profile_entity_check = (
    df_train[['profile_id', 'entity_id']]
    .drop_duplicates()
    .groupby('profile_id')['entity_id']
    .nunique()
)

bad_profiles = profile_entity_check[profile_entity_check > 1].index
profile_df = profile_df[~profile_df['profile_id'].isin(bad_profiles)].reset_index(drop=True)

# Индексы узлов

In [10]:
profile_ids = profile_df['profile_id'].tolist()
profile2idx = {p: i for i, p in enumerate(profile_ids)}

attribute_cols = {
    'domain': 'domain_set',
    'phone': 'phone_set',
    'device': 'device_set',
    'city': 'city_set',
    'region': 'region_set',
}

attr_value_to_idx = {}
for attr_type, set_col in attribute_cols.items():
    values = sorted(set().union(*profile_df[set_col].tolist()))
    attr_value_to_idx[attr_type] = {v: i for i, v in enumerate(values)}

# Построение HeteroData

In [11]:
data = HeteroData()

# profile node features
profile_num_features = [
    'row_count', 'n_email', 'n_phone', 'n_domain', 'n_device', 'n_city', 'n_region',
    'local_hour_mean', 'visit_count_mean', 'city_population_mean',
    'fs_is_phone_mean', 'fs_is_gmail_mean', 'fs_is_yandex_mean',
    'fs_is_man_mean', 'fs_is_woman_mean'
]

profile_x = profile_df[profile_num_features].copy()
profile_x = profile_x.fillna(0.0)
data['profile'].x = torch.tensor(profile_x.values, dtype=torch.float)

# attribute node features: пока простые частотные
for attr_type, mapper in attr_value_to_idx.items():
    num_nodes = len(mapper)
    data[attr_type].x = torch.ones((num_nodes, 1), dtype=torch.float)

# edges
for attr_type, set_col in attribute_cols.items():
    src = []
    dst = []
    mapper = attr_value_to_idx[attr_type]

    for _, row in profile_df[['profile_id', set_col]].iterrows():
        p_idx = profile2idx[row['profile_id']]
        for v in row[set_col]:
            if v in mapper:
                src.append(p_idx)
                dst.append(mapper[v])

    edge_index = torch.tensor([src, dst], dtype=torch.long)
    data['profile', f'has_{attr_type}', attr_type].edge_index = edge_index
    data[attr_type, f'rev_has_{attr_type}', 'profile'].edge_index = edge_index.flip(0)

data

HeteroData(
  profile={ x=[37069, 15] },
  domain={ x=[517, 1] },
  phone={ x=[2714, 1] },
  device={ x=[3, 1] },
  city={ x=[1357, 1] },
  region={ x=[305, 1] },
  (profile, has_domain, domain)={ edge_index=[2, 37069] },
  (domain, rev_has_domain, profile)={ edge_index=[2, 37069] },
  (profile, has_phone, phone)={ edge_index=[2, 2757] },
  (phone, rev_has_phone, profile)={ edge_index=[2, 2757] },
  (profile, has_device, device)={ edge_index=[2, 35887] },
  (device, rev_has_device, profile)={ edge_index=[2, 35887] },
  (profile, has_city, city)={ edge_index=[2, 29911] },
  (city, rev_has_city, profile)={ edge_index=[2, 29911] },
  (profile, has_region, region)={ edge_index=[2, 33385] },
  (region, rev_has_region, profile)={ edge_index=[2, 33385] }
)

#  Модель GraphSAGE

In [12]:
class GNNEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

encoder = GNNEncoder(hidden_channels=64, out_channels=64)
model = to_hetero(encoder, data.metadata(), aggr='sum')

# Обучение self-supervised / proxy task
Так как у нас цель потом на парах профилей, самый практичный каркас — обучить embeddings через proxy-задачу реконструкции графовых связей profile -> attribute. То есть положительные ребра — реальные связи, отрицательные — случайные связи. Это нормальная схема для graph representation learning.

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

def edge_score(z_src, z_dst, edge_index):
    return (z_src[edge_index[0]] * z_dst[edge_index[1]]).sum(dim=1)

def sample_negative_edges(num_src, num_dst, num_samples, device):
    src = torch.randint(0, num_src, (num_samples,), device=device)
    dst = torch.randint(0, num_dst, (num_samples,), device=device)
    return torch.stack([src, dst], dim=0)

edge_types_to_train = [
    ('profile', 'has_domain', 'domain'),
    ('profile', 'has_phone', 'phone'),
    ('profile', 'has_device', 'device'),
    ('profile', 'has_city', 'city'),
    ('profile', 'has_region', 'region'),
]

for epoch in range(1, 101):
    model.train()
    optimizer.zero_grad()

    z_dict = model(data.x_dict, data.edge_index_dict)

    total_loss = 0
    for et in edge_types_to_train:
        pos_edge_index = data[et].edge_index
        num_pos = pos_edge_index.size(1)

        neg_edge_index = sample_negative_edges(
            z_dict[et[0]].size(0),
            z_dict[et[2]].size(0),
            num_pos,
            device
        )

        pos_score = edge_score(z_dict[et[0]], z_dict[et[2]], pos_edge_index)
        neg_score = edge_score(z_dict[et[0]], z_dict[et[2]], neg_edge_index)

        pos_loss = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
        neg_loss = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))

        total_loss = total_loss + pos_loss + neg_loss

    total_loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d}, loss={total_loss.item():.4f}")

Epoch 010, loss=2262574563328.0000
Epoch 020, loss=354473902080.0000
Epoch 030, loss=394116890624.0000
Epoch 040, loss=189921902592.0000
Epoch 050, loss=123082563584.0000
Epoch 060, loss=114736742400.0000
Epoch 070, loss=229670879232.0000
Epoch 080, loss=217123618816.0000
Epoch 090, loss=264117043200.0000
Epoch 100, loss=126743846912.0000


In [14]:
model.eval()
with torch.no_grad():
    z_dict = model(data.x_dict, data.edge_index_dict)

profile_emb = z_dict['profile'].cpu().numpy()

emb_df = pd.DataFrame(profile_emb, columns=[f'emb_{i}' for i in range(profile_emb.shape[1])])
emb_df['profile_id'] = profile_df['profile_id'].values
profile_df = profile_df.merge(emb_df, on='profile_id', how='left')

# Positive pairs
Теперь строим положительные пары (profile_id_1, profile_id_2) внутри одного entity_id. Это и есть target = 1 для задачи same-person.

In [15]:
candidate_pairs = set()

for attr_type, set_col in attribute_cols.items():
    inv_index = defaultdict(list)

    for _, row in profile_df[['profile_id', set_col]].iterrows():
        p = row['profile_id']
        for v in row[set_col]:
            inv_index[(attr_type, v)].append(p)

    for _, prof_list in inv_index.items():
        prof_list = sorted(set(prof_list))
        if 2 <= len(prof_list) <= 200:
            for a, b in combinations(prof_list, 2):
                candidate_pairs.add((a, b))

```python
emb_cols = [c for c in profile_df.columns if c.startswith('emb_')]
X_emb = profile_df[emb_cols].values

nn = NearestNeighbors(n_neighbors=20, metric='cosine')
nn.fit(X_emb)

distances, indices = nn.kneighbors(X_emb)

profiles = profile_df['profile_id'].tolist()
for i, neigh_idx in enumerate(indices):
    p1 = profiles[i]
    for j in neigh_idx[1:]:
        p2 = profiles[j]
        a, b = sorted([p1, p2])
        candidate_pairs.add((a, b))

In [16]:
cand_pairs = pd.DataFrame(list(candidate_pairs), columns=['profile_id_1', 'profile_id_2'])

# Разметка кандидатов

In [17]:
profile_to_entity = profile_df.set_index('profile_id')['entity_id'].to_dict()

cand_pairs['label'] = (
    cand_pairs['profile_id_1'].map(profile_to_entity) ==
    cand_pairs['profile_id_2'].map(profile_to_entity)
).astype(int)

cand_pairs['is_positive_candidate'] = cand_pairs['label']
cand_pairs['label'].value_counts()

label
0    755679
1      1776
Name: count, dtype: int64

In [18]:
pos_df = cand_pairs[cand_pairs['label'] == 1]
neg_df = cand_pairs[cand_pairs['label'] == 0].sample(
    n=min(len(cand_pairs[cand_pairs['label'] == 0]), len(pos_df) * 5),
    random_state=42
)

pairs_df = pd.concat([pos_df, neg_df], ignore_index=True)

# Pairwise features
Теперь делаем обычные признаки пары + embedding similarity. Это обычно дает сильнее результат, чем только embeddings.



In [19]:
def same_any(a, b):
    return int(len(a & b) > 0)

def intersect_size(a, b):
    return len(a & b)

def jaccard(a, b):
    u = len(a | b)
    return len(a & b) / u if u else 0.0

pairs_df = pairs_df.merge(
    profile_df.add_suffix('_1'),
    left_on='profile_id_1',
    right_on='profile_id_1',
    how='left'
)

pairs_df = pairs_df.merge(
    profile_df.add_suffix('_2'),
    left_on='profile_id_2',
    right_on='profile_id_2',
    how='left'
)

In [20]:
pairs_df['same_phone_any'] = pairs_df.apply(lambda r: same_any(r.phone_set_1, r.phone_set_2), axis=1)
pairs_df['same_email_any'] = pairs_df.apply(lambda r: same_any(r.email_set_1, r.email_set_2), axis=1)
pairs_df['same_domain_any'] = pairs_df.apply(lambda r: same_any(r.domain_set_1, r.domain_set_2), axis=1)
pairs_df['same_device_any'] = pairs_df.apply(lambda r: same_any(r.device_set_1, r.device_set_2), axis=1)
pairs_df['same_city_any'] = pairs_df.apply(lambda r: same_any(r.city_set_1, r.city_set_2), axis=1)
pairs_df['same_region_any'] = pairs_df.apply(lambda r: same_any(r.region_set_1, r.region_set_2), axis=1)
pairs_df['same_browser_any'] = pairs_df.apply(lambda r: same_any(r.browser_set_1, r.browser_set_2), axis=1)
pairs_df['same_os_any'] = pairs_df.apply(lambda r: same_any(r.os_set_1, r.os_set_2), axis=1)

pairs_df['common_phone_cnt'] = pairs_df.apply(lambda r: intersect_size(r.phone_set_1, r.phone_set_2), axis=1)
pairs_df['common_email_cnt'] = pairs_df.apply(lambda r: intersect_size(r.email_set_1, r.email_set_2), axis=1)
pairs_df['common_domain_cnt'] = pairs_df.apply(lambda r: intersect_size(r.domain_set_1, r.domain_set_2), axis=1)

pairs_df['domain_jaccard'] = pairs_df.apply(lambda r: jaccard(r.domain_set_1, r.domain_set_2), axis=1)
pairs_df['device_jaccard'] = pairs_df.apply(lambda r: jaccard(r.device_set_1, r.device_set_2), axis=1)
pairs_df['city_jaccard'] = pairs_df.apply(lambda r: jaccard(r.city_set_1, r.city_set_2), axis=1)

pairs_df['row_count_diff'] = (pairs_df['row_count_1'] - pairs_df['row_count_2']).abs()
pairs_df['local_hour_mean_diff'] = (pairs_df['local_hour_mean_1'] - pairs_df['local_hour_mean_2']).abs()
pairs_df['visit_count_mean_diff'] = (pairs_df['visit_count_mean_1'] - pairs_df['visit_count_mean_2']).abs()
pairs_df['city_population_mean_diff'] = (pairs_df['city_population_mean_1'] - pairs_df['city_population_mean_2']).abs()

In [21]:
emb_cols_1 = [c for c in pairs_df.columns if c.startswith('emb_') and c.endswith('_1')]
emb_cols_2 = [c for c in pairs_df.columns if c.startswith('emb_') and c.endswith('_2')]

E1 = pairs_df[emb_cols_1].values
E2 = pairs_df[emb_cols_2].values

dot = (E1 * E2).sum(axis=1)
norm1 = np.linalg.norm(E1, axis=1)
norm2 = np.linalg.norm(E2, axis=1)

pairs_df['emb_dot'] = dot
pairs_df['emb_cosine'] = dot / (norm1 * norm2 + 1e-9)
pairs_df['emb_l2'] = np.linalg.norm(E1 - E2, axis=1)
pairs_df['emb_l1'] = np.abs(E1 - E2).sum(axis=1)

for i in range(E1.shape[1]):
    pairs_df[f'emb_absdiff_{i}'] = np.abs(E1[:, i] - E2[:, i])
    pairs_df[f'emb_mul_{i}'] = E1[:, i] * E2[:, i]

/tmp/ipykernel_47744/1156264803.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'emb_absdiff_{i}'] = np.abs(E1[:, i] - E2[:, i])
/tmp/ipykernel_47744/1156264803.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'emb_mul_{i}'] = E1[:, i] * E2[:, i]
/tmp/ipykernel_47744/1156264803.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To

# Обучение CatBoost

In [22]:
drop_cols = [
    'profile_id_1', 'profile_id_2', 'label',
    'entity_id_1', 'entity_id_2',
    'email_set_1', 'phone_set_1', 'domain_set_1', 'device_set_1', 'city_set_1',
    'region_set_1', 'browser_set_1', 'os_set_1', 'sex_set_1',
    'email_set_2', 'phone_set_2', 'domain_set_2', 'device_set_2', 'city_set_2',
    'region_set_2', 'browser_set_2', 'os_set_2', 'sex_set_2',
]

feature_cols = [c for c in pairs_df.columns if c not in drop_cols]

X = pairs_df[feature_cols].copy()
y = pairs_df['label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_cb = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=50
)

model_cb.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

val_pred = model_cb.predict_proba(X_val)[:, 1]
print("ROC-AUC:", roc_auc_score(y_val, val_pred))
print("PR-AUC:", average_precision_score(y_val, val_pred))

0:	test: 1.0000000	best: 1.0000000 (0)	total: 55.6ms	remaining: 27.7s
50:	test: 1.0000000	best: 1.0000000 (0)	total: 446ms	remaining: 3.93s
100:	test: 1.0000000	best: 1.0000000 (0)	total: 855ms	remaining: 3.38s
150:	test: 1.0000000	best: 1.0000000 (0)	total: 1.25s	remaining: 2.88s
200:	test: 1.0000000	best: 1.0000000 (0)	total: 1.66s	remaining: 2.47s
250:	test: 1.0000000	best: 1.0000000 (0)	total: 2.05s	remaining: 2.03s
300:	test: 1.0000000	best: 1.0000000 (0)	total: 2.44s	remaining: 1.61s
350:	test: 1.0000000	best: 1.0000000 (0)	total: 2.88s	remaining: 1.22s
400:	test: 1.0000000	best: 1.0000000 (0)	total: 3.28s	remaining: 809ms
450:	test: 1.0000000	best: 1.0000000 (0)	total: 3.7s	remaining: 402ms
499:	test: 1.0000000	best: 1.0000000 (0)	total: 4.1s	remaining: 0us

bestTest = 1
bestIteration = 0

Shrink model to first 1 iterations.
ROC-AUC: 1.0
PR-AUC: 1.0


In [23]:
import numpy as np
import pandas as pd

def build_pair_features_from_df(pairs_df, df, feature_cols):
    """
    pairs_df: колонки ['profile_id_1', 'profile_id_2', 'label' (опционально)]
    df: исходный датафрейм с profile_id, entity_id, *_set и числовыми фичами
    feature_cols: список колонок, на которых обучался model_cb
    """
    base = df.copy()

    set_cols = ['email_set', 'phone_set', 'domain_set', 'device_set',
                'city_set', 'region_set']
    set_cols = [c for c in set_cols if c in base.columns]

    num_cols = [c for c in ['row_count', 'localhour_mean',
                            'visitcount_mean', 'citypopulation_mean']
                if c in base.columns]

    emb_cols = [c for c in base.columns if c.startswith('emb')]

    # левая / правая части
    left = base.add_suffix('_l').rename(columns={'profile_id_l': 'profile_id_1'})
    right = base.add_suffix('_r').rename(columns={'profile_id_r': 'profile_id_2'})

    df_pairs = (pairs_df
                .merge(left, on='profile_id_1', how='left')
                .merge(right, on='profile_id_2', how='left'))

    def to_set(x):
        if isinstance(x, set):
            return x
        if isinstance(x, (list, tuple)):
            return set(x)
        return set()

    # set-based признаки
    for c in set_cols:
        cl, cr = f'{c}_l', f'{c}_r'
        l_sets = df_pairs[cl].apply(to_set)
        r_sets = df_pairs[cr].apply(to_set)
        df_pairs[f'same_{c}'] = [int(len(a & b) > 0) for a, b in zip(l_sets, r_sets)]
        df_pairs[f'jaccard_{c}'] = [
            (len(a & b) / len(a | b)) if len(a | b) else 0.0
            for a, b in zip(l_sets, r_sets)
        ]

    # числовые разности
    for c in num_cols:
        cl, cr = f'{c}_l', f'{c}_r'
        df_pairs[f'diff_{c}'] = (df_pairs[cl].fillna(0) - df_pairs[cr].fillna(0)).abs()

    # эмбеддинги, если есть
    emb_cols_l = [f'{c}_l' for c in emb_cols]
    emb_cols_r = [f'{c}_r' for c in emb_cols]
    if emb_cols_l and emb_cols_r:
        E1 = df_pairs[emb_cols_l].fillna(0).values
        E2 = df_pairs[emb_cols_r].fillna(0).values
        norm1 = np.linalg.norm(E1, axis=1) + 1e-9
        norm2 = np.linalg.norm(E2, axis=1) + 1e-9
        df_pairs['emb_cosine'] = (E1 * E2).sum(axis=1) / (norm1 * norm2)
        df_pairs['emb_dot'] = (E1 * E2).sum(axis=1)
        df_pairs['emb_l2'] = np.linalg.norm(E1 - E2, axis=1)
        df_pairs['emb_l1'] = np.abs(E1 - E2).sum(axis=1)
        abs_diff = np.abs(E1 - E2)
        mul_diff = E1 * E2
        for i in range(E1.shape[1]):
            df_pairs[f'emb_absdiff_{i}'] = abs_diff[:, i]
            df_pairs[f'emb_mul_{i}'] = mul_diff[:, i]

    # гарантируем наличие всех feature_cols
    for col in feature_cols:
        if col not in df_pairs.columns:
            df_pairs[col] = 0

    X = df_pairs[feature_cols]
    y = df_pairs['label'] if 'label' in df_pairs.columns else None
    return X, y, df_pairs

In [24]:
import os
import json
import pickle
import random
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, average_precision_score

from catboost import CatBoostClassifier

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero


# =========================================================
# 0. Utils
# =========================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def to_set(series):
    vals = series.dropna().astype(str).unique().tolist()
    vals = [v for v in vals if v not in ['nan', 'None', '']]
    return set(vals)


def mode_or_nan(series):
    s = series.dropna().astype(str)
    s = s[~s.isin(['nan', 'None', ''])]
    if len(s) == 0:
        return np.nan
    return s.mode().iat[0]


def same_any(a, b):
    return int(len(a & b) > 0)


def intersect_size(a, b):
    return len(a & b)


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def overlap_coef(a, b):
    denom = min(len(a), len(b))
    return len(a & b) / denom if denom else 0.0


def cosine_similarity_matrix_topk(X, n_neighbors=20):
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine')
    nn.fit(X)
    distances, indices = nn.kneighbors(X)
    return distances, indices


# =========================================================
# 1. Profile aggregation
# =========================================================

def build_profile_df(df: pd.DataFrame, split: str = 'train') -> pd.DataFrame:
    df_part = df[df['split'] == split].copy()

    cast_cols = [
        'profile_id', 'entity_id', 'email', 'phone', 'email_domain',
        'device', 'city_name', 'region_code', 'browser', 'osfamily', 'sex'
    ]
    for col in cast_cols:
        if col in df_part.columns:
            df_part[col] = df_part[col].astype(str)
            df_part.loc[df_part[col].isin(['nan', 'None']), col] = np.nan

    agg_dict = {
        'entity_id': ('entity_id', mode_or_nan),
        'row_count': ('profile_id', 'size'),
        'email_set': ('email', to_set),
        'phone_set': ('phone', to_set),
        'domain_set': ('email_domain', to_set),
        'device_set': ('device', to_set),
        'city_set': ('city_name', to_set),
        'region_set': ('region_code', to_set),
        'browser_set': ('browser', to_set),
        'os_set': ('osfamily', to_set),
        'sex_set': ('sex', to_set),
    }

    optional_numeric = [
        'local_hour', 'visit_count', 'city_population',
        'fs_is_phone', 'fs_is_gmail', 'fs_is_yandex',
        'fs_is_man', 'fs_is_woman'
    ]
    for col in optional_numeric:
        if col in df_part.columns:
            agg_dict[f'{col}_mean'] = (col, 'mean')

    profile_df = (
        df_part.groupby('profile_id')
        .agg(**agg_dict)
        .reset_index()
    )

    profile_df['n_email'] = profile_df['email_set'].apply(len)
    profile_df['n_phone'] = profile_df['phone_set'].apply(len)
    profile_df['n_domain'] = profile_df['domain_set'].apply(len)
    profile_df['n_device'] = profile_df['device_set'].apply(len)
    profile_df['n_city'] = profile_df['city_set'].apply(len)
    profile_df['n_region'] = profile_df['region_set'].apply(len)
    profile_df['n_browser'] = profile_df['browser_set'].apply(len)
    profile_df['n_os'] = profile_df['os_set'].apply(len)

    profile_entity_check = (
        df_part[['profile_id', 'entity_id']]
        .drop_duplicates()
        .groupby('profile_id')['entity_id']
        .nunique()
    )
    bad_profiles = profile_entity_check[profile_entity_check > 1].index
    if len(bad_profiles) > 0:
        profile_df = profile_df[~profile_df['profile_id'].isin(bad_profiles)].reset_index(drop=True)

    return profile_df


# =========================================================
# 2. Heterograph
# =========================================================

def fit_graph_schema(
    profile_df: pd.DataFrame,
    attribute_cols=None,
    max_freq_per_value: int = 5000
):
    if attribute_cols is None:
        attribute_cols = {
            'domain': 'domain_set',
            'phone': 'phone_set',
            'device': 'device_set',
            'city': 'city_set',
            'region': 'region_set',
        }

    attr_value_counts = {}
    attr_value_to_idx = {}

    for attr_type, set_col in attribute_cols.items():
        counter = defaultdict(int)
        for vals in profile_df[set_col]:
            for v in vals:
                counter[v] += 1

        filtered_values = sorted([v for v, c in counter.items() if c <= max_freq_per_value])
        attr_value_counts[attr_type] = dict(counter)
        attr_value_to_idx[attr_type] = {v: i for i, v in enumerate(filtered_values)}

    profile_feature_cols = [
        c for c in [
            'row_count', 'n_email', 'n_phone', 'n_domain', 'n_device',
            'n_city', 'n_region', 'n_browser', 'n_os',
            'local_hour_mean', 'visit_count_mean', 'city_population_mean',
            'fs_is_phone_mean', 'fs_is_gmail_mean', 'fs_is_yandex_mean',
            'fs_is_man_mean', 'fs_is_woman_mean'
        ] if c in profile_df.columns
    ]

    profile_feature_stats = {}
    tmp = profile_df[profile_feature_cols].copy().fillna(0.0)
    for c in profile_feature_cols:
        profile_feature_stats[c] = {
            'mean': float(tmp[c].mean()),
            'std': float(tmp[c].std(ddof=0) + 1e-9)
        }

    attr_feature_stats = {}
    for attr_type, mapper in attr_value_to_idx.items():
        freqs = [attr_value_counts[attr_type][v] for v in mapper.keys()]
        x_attr = pd.DataFrame({
            'freq': freqs,
            'log_freq': np.log1p(freqs),
        })
        attr_feature_stats[attr_type] = {
            'mean': x_attr.mean().to_dict(),
            'std': (x_attr.std(ddof=0) + 1e-9).to_dict()
        }

    schema = {
        'attribute_cols': attribute_cols,
        'attr_value_to_idx': attr_value_to_idx,
        'attr_value_counts': attr_value_counts,
        'profile_feature_cols': profile_feature_cols,
        'profile_feature_stats': profile_feature_stats,
        'attr_feature_stats': attr_feature_stats,
    }
    return schema


def build_hetero_graph_from_schema(profile_df: pd.DataFrame, schema: dict):
    attribute_cols = schema['attribute_cols']
    attr_value_to_idx = schema['attr_value_to_idx']
    attr_value_counts = schema['attr_value_counts']
    profile_feature_cols = schema['profile_feature_cols']
    profile_feature_stats = schema['profile_feature_stats']
    attr_feature_stats = schema['attr_feature_stats']

    profile_ids = profile_df['profile_id'].tolist()
    profile2idx = {p: i for i, p in enumerate(profile_ids)}

    data = HeteroData()

    # profile features normalized with train stats
    profile_x = profile_df[profile_feature_cols].copy().fillna(0.0)
    for c in profile_feature_cols:
        mean_ = profile_feature_stats[c]['mean']
        std_ = profile_feature_stats[c]['std']
        profile_x[c] = (profile_x[c] - mean_) / std_
    data['profile'].x = torch.tensor(profile_x.values, dtype=torch.float)

    # attr features
    for attr_type, mapper in attr_value_to_idx.items():
        freqs = [attr_value_counts[attr_type].get(v, 1) for v in mapper.keys()]
        x_attr = pd.DataFrame({
            'freq': freqs,
            'log_freq': np.log1p(freqs),
        })
        for col in ['freq', 'log_freq']:
            mean_ = attr_feature_stats[attr_type]['mean'][col]
            std_ = attr_feature_stats[attr_type]['std'][col]
            x_attr[col] = (x_attr[col] - mean_) / std_
        data[attr_type].x = torch.tensor(x_attr.values, dtype=torch.float)

    # edges
    for attr_type, set_col in attribute_cols.items():
        src, dst = [], []
        mapper = attr_value_to_idx[attr_type]

        for _, row in profile_df[['profile_id', set_col]].iterrows():
            p_idx = profile2idx[row['profile_id']]
            for v in row[set_col]:
                if v in mapper:
                    src.append(p_idx)
                    dst.append(mapper[v])

        if len(src) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor([src, dst], dtype=torch.long)

        data['profile', f'has_{attr_type}', attr_type].edge_index = edge_index
        data[attr_type, f'rev_has_{attr_type}', 'profile'].edge_index = edge_index.flip(0)

    meta = {
        'profile2idx': profile2idx,
        'profile_ids': profile_ids,
    }
    return data, meta


# =========================================================
# 3. GraphSAGE
# =========================================================

class GNNEncoder(nn.Module):
    def __init__(self, hidden_channels=64, out_channels=64):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x


def build_graphsage_model(data: HeteroData, hidden_channels=64, out_channels=64):
    encoder = GNNEncoder(hidden_channels=hidden_channels, out_channels=out_channels)
    model = to_hetero(encoder, data.metadata(), aggr='sum')
    return model


def edge_score(z_src, z_dst, edge_index):
    if edge_index.size(1) == 0:
        return torch.empty((0,), device=z_src.device)
    return (z_src[edge_index[0]] * z_dst[edge_index[1]]).sum(dim=1)


def sample_negative_edges(num_src, num_dst, num_samples, device):
    src = torch.randint(0, num_src, (num_samples,), device=device)
    dst = torch.randint(0, num_dst, (num_samples,), device=device)
    return torch.stack([src, dst], dim=0)


def train_graphsage(
    data: HeteroData,
    hidden_channels=64,
    out_channels=64,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    device=None,
    verbose=10
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = build_graphsage_model(data, hidden_channels, out_channels).to(device)
    data = data.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    edge_types_to_train = [
        et for et in data.edge_types
        if et[0] == 'profile' and et[2] != 'profile'
    ]

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        z_dict = model(data.x_dict, data.edge_index_dict)
        total_loss = 0.0

        for et in edge_types_to_train:
            pos_edge_index = data[et].edge_index
            num_pos = pos_edge_index.size(1)
            if num_pos == 0:
                continue

            neg_edge_index = sample_negative_edges(
                z_dict[et[0]].size(0),
                z_dict[et[2]].size(0),
                num_pos,
                device
            )

            pos_score = edge_score(z_dict[et[0]], z_dict[et[2]], pos_edge_index)
            neg_score = edge_score(z_dict[et[0]], z_dict[et[2]], neg_edge_index)

            pos_loss = F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
            neg_loss = F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
            total_loss = total_loss + pos_loss + neg_loss

        total_loss.backward()
        optimizer.step()

        if verbose and epoch % verbose == 0:
            print(f'Epoch {epoch:03d} | loss={total_loss.item():.4f}')

    return model


def infer_profile_embeddings(model, data: HeteroData, device=None):
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    data = data.to(device)

    with torch.no_grad():
        z_dict = model(data.x_dict, data.edge_index_dict)

    return z_dict['profile'].cpu().numpy()


def attach_embeddings(profile_df: pd.DataFrame, profile_emb: np.ndarray) -> pd.DataFrame:
    emb_df = pd.DataFrame(profile_emb, columns=[f'emb_{i}' for i in range(profile_emb.shape[1])])
    emb_df['profile_id'] = profile_df['profile_id'].values
    return profile_df.merge(emb_df, on='profile_id', how='left')


# =========================================================
# 4. Candidate generation
# =========================================================

def generate_positive_pairs(profile_df: pd.DataFrame) -> pd.DataFrame:
    pos_pairs = []
    for entity_id, grp in profile_df.groupby('entity_id'):
        profiles = sorted(grp['profile_id'].tolist())
        if len(profiles) >= 2:
            pos_pairs.extend([(a, b, 1) for a, b in combinations(profiles, 2)])
    return pd.DataFrame(pos_pairs, columns=['profile_id_1', 'profile_id_2', 'label'])


def generate_candidate_pairs_2hop(
    profile_df: pd.DataFrame,
    attribute_cols: dict,
    max_group_size: int = 200
):
    candidate_pairs = set()

    for attr_type, set_col in attribute_cols.items():
        inv_index = defaultdict(list)
        for _, row in profile_df[['profile_id', set_col]].iterrows():
            p = row['profile_id']
            for v in row[set_col]:
                inv_index[(attr_type, v)].append(p)

        for _, prof_list in inv_index.items():
            prof_list = sorted(set(prof_list))
            if 2 <= len(prof_list) <= max_group_size:
                for a, b in combinations(prof_list, 2):
                    candidate_pairs.add((a, b))

    return candidate_pairs


def generate_candidate_pairs_knn(
    profile_df: pd.DataFrame,
    emb_prefix='emb_',
    n_neighbors=20,
    metric='cosine'
):
    emb_cols = [c for c in profile_df.columns if c.startswith(emb_prefix)]
    X_emb = profile_df[emb_cols].values
    profiles = profile_df['profile_id'].tolist()

    nn = NearestNeighbors(n_neighbors=n_neighbors, metric=metric)
    nn.fit(X_emb)
    _, indices = nn.kneighbors(X_emb)

    candidate_pairs = set()
    for i, neigh_idx in enumerate(indices):
        p1 = profiles[i]
        for j in neigh_idx[1:]:
            p2 = profiles[j]
            a, b = sorted([p1, p2])
            candidate_pairs.add((a, b))

    return candidate_pairs


def label_candidate_pairs(candidate_pairs, profile_df: pd.DataFrame) -> pd.DataFrame:
    pairs_df = pd.DataFrame(list(candidate_pairs), columns=['profile_id_1', 'profile_id_2'])
    profile_to_entity = profile_df.set_index('profile_id')['entity_id'].to_dict()
    pairs_df['label'] = (
        pairs_df['profile_id_1'].map(profile_to_entity) ==
        pairs_df['profile_id_2'].map(profile_to_entity)
    ).astype(int)
    return pairs_df


def sample_pairs_for_training(
    pairs_df: pd.DataFrame,
    neg_to_pos_ratio: int = 5,
    random_state: int = 42
):
    pos_df = pairs_df[pairs_df['label'] == 1].copy()
    neg_df = pairs_df[pairs_df['label'] == 0].copy()

    if len(pos_df) == 0:
        raise ValueError('Нет положительных candidate pairs. Проверьте генерацию кандидатов.')

    n_neg = min(len(neg_df), len(pos_df) * neg_to_pos_ratio)
    neg_df = neg_df.sample(n=n_neg, random_state=random_state) if len(neg_df) > n_neg else neg_df

    out = pd.concat([pos_df, neg_df], ignore_index=True)
    out = out.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return out


# =========================================================
# 5. Pair features
# =========================================================

def build_pair_features(pairs_df: pd.DataFrame, profile_df: pd.DataFrame) -> pd.DataFrame:
    pairs_df = pairs_df.merge(
        profile_df.add_suffix('_1'),
        left_on='profile_id_1',
        right_on='profile_id_1',
        how='left'
    )
    pairs_df = pairs_df.merge(
        profile_df.add_suffix('_2'),
        left_on='profile_id_2',
        right_on='profile_id_2',
        how='left'
    )

    pairs_df['same_phone_any'] = pairs_df.apply(lambda r: same_any(r.phone_set_1, r.phone_set_2), axis=1)
    pairs_df['same_email_any'] = pairs_df.apply(lambda r: same_any(r.email_set_1, r.email_set_2), axis=1)
    pairs_df['same_domain_any'] = pairs_df.apply(lambda r: same_any(r.domain_set_1, r.domain_set_2), axis=1)
    pairs_df['same_device_any'] = pairs_df.apply(lambda r: same_any(r.device_set_1, r.device_set_2), axis=1)
    pairs_df['same_city_any'] = pairs_df.apply(lambda r: same_any(r.city_set_1, r.city_set_2), axis=1)
    pairs_df['same_region_any'] = pairs_df.apply(lambda r: same_any(r.region_set_1, r.region_set_2), axis=1)
    pairs_df['same_browser_any'] = pairs_df.apply(lambda r: same_any(r.browser_set_1, r.browser_set_2), axis=1)
    pairs_df['same_os_any'] = pairs_df.apply(lambda r: same_any(r.os_set_1, r.os_set_2), axis=1)
    pairs_df['same_sex_any'] = pairs_df.apply(lambda r: same_any(r.sex_set_1, r.sex_set_2), axis=1)

    pairs_df['common_phone_cnt'] = pairs_df.apply(lambda r: intersect_size(r.phone_set_1, r.phone_set_2), axis=1)
    pairs_df['common_email_cnt'] = pairs_df.apply(lambda r: intersect_size(r.email_set_1, r.email_set_2), axis=1)
    pairs_df['common_domain_cnt'] = pairs_df.apply(lambda r: intersect_size(r.domain_set_1, r.domain_set_2), axis=1)
    pairs_df['common_device_cnt'] = pairs_df.apply(lambda r: intersect_size(r.device_set_1, r.device_set_2), axis=1)
    pairs_df['common_city_cnt'] = pairs_df.apply(lambda r: intersect_size(r.city_set_1, r.city_set_2), axis=1)
    pairs_df['common_region_cnt'] = pairs_df.apply(lambda r: intersect_size(r.region_set_1, r.region_set_2), axis=1)

    pairs_df['domain_jaccard'] = pairs_df.apply(lambda r: jaccard(r.domain_set_1, r.domain_set_2), axis=1)
    pairs_df['device_jaccard'] = pairs_df.apply(lambda r: jaccard(r.device_set_1, r.device_set_2), axis=1)
    pairs_df['city_jaccard'] = pairs_df.apply(lambda r: jaccard(r.city_set_1, r.city_set_2), axis=1)
    pairs_df['phone_jaccard'] = pairs_df.apply(lambda r: jaccard(r.phone_set_1, r.phone_set_2), axis=1)
    pairs_df['email_jaccard'] = pairs_df.apply(lambda r: jaccard(r.email_set_1, r.email_set_2), axis=1)

    pairs_df['domain_overlap'] = pairs_df.apply(lambda r: overlap_coef(r.domain_set_1, r.domain_set_2), axis=1)
    pairs_df['device_overlap'] = pairs_df.apply(lambda r: overlap_coef(r.device_set_1, r.device_set_2), axis=1)
    pairs_df['city_overlap'] = pairs_df.apply(lambda r: overlap_coef(r.city_set_1, r.city_set_2), axis=1)

    numeric_base_cols = [
        'row_count', 'n_email', 'n_phone', 'n_domain', 'n_device',
        'n_city', 'n_region', 'n_browser', 'n_os',
        'local_hour_mean', 'visit_count_mean', 'city_population_mean',
        'fs_is_phone_mean', 'fs_is_gmail_mean', 'fs_is_yandex_mean',
        'fs_is_man_mean', 'fs_is_woman_mean'
    ]

    for col in numeric_base_cols:
        c1 = f'{col}_1'
        c2 = f'{col}_2'
        if c1 in pairs_df.columns and c2 in pairs_df.columns:
            pairs_df[f'{col}_diff'] = (pairs_df[c1] - pairs_df[c2]).abs()
            pairs_df[f'{col}_min'] = pairs_df[[c1, c2]].min(axis=1)
            pairs_df[f'{col}_max'] = pairs_df[[c1, c2]].max(axis=1)

    emb_cols_1 = sorted([c for c in pairs_df.columns if c.startswith('emb_') and c.endswith('_1')])
    emb_cols_2 = sorted([c for c in pairs_df.columns if c.startswith('emb_') and c.endswith('_2')])

    if len(emb_cols_1) > 0 and len(emb_cols_2) > 0:
        E1 = pairs_df[emb_cols_1].values
        E2 = pairs_df[emb_cols_2].values

        dot = (E1 * E2).sum(axis=1)
        norm1 = np.linalg.norm(E1, axis=1)
        norm2 = np.linalg.norm(E2, axis=1)

        pairs_df['emb_dot'] = dot
        pairs_df['emb_cosine'] = dot / (norm1 * norm2 + 1e-9)
        pairs_df['emb_l2'] = np.linalg.norm(E1 - E2, axis=1)
        pairs_df['emb_l1'] = np.abs(E1 - E2).sum(axis=1)

        for i in range(E1.shape[1]):
            pairs_df[f'emb_absdiff_{i}'] = np.abs(E1[:, i] - E2[:, i])
            pairs_df[f'emb_mul_{i}'] = E1[:, i] * E2[:, i]

    return pairs_df


# =========================================================
# 6. CatBoost
# =========================================================

def get_feature_columns(pairs_feat_df: pd.DataFrame):
    drop_cols = [
        'profile_id_1', 'profile_id_2', 'label',
        'entity_id_1', 'entity_id_2',
        'email_set_1', 'phone_set_1', 'domain_set_1', 'device_set_1', 'city_set_1',
        'region_set_1', 'browser_set_1', 'os_set_1', 'sex_set_1',
        'email_set_2', 'phone_set_2', 'domain_set_2', 'device_set_2', 'city_set_2',
        'region_set_2', 'browser_set_2', 'os_set_2', 'sex_set_2',
    ]
    feature_cols = [c for c in pairs_feat_df.columns if c not in drop_cols]
    return feature_cols


def train_catboost_pair_classifier(
    pairs_feat_df: pd.DataFrame,
    test_size=0.2,
    random_state=42,
    iterations=500,
    depth=6,
    learning_rate=0.05,
    verbose=100
):
    feature_cols = get_feature_columns(pairs_feat_df)
    X = pairs_feat_df[feature_cols].copy()
    y = pairs_feat_df['label'].values

    X = X.replace([np.inf, -np.inf], np.nan).fillna(-999.0)

    X_train, X_val, y_train, y_val = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    model = CatBoostClassifier(
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        loss_function='Logloss',
        eval_metric='AUC',
        random_seed=random_state,
        verbose=verbose
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    val_pred = model.predict_proba(X_val)[:, 1]
    metrics = {
        'roc_auc': float(roc_auc_score(y_val, val_pred)),
        'pr_auc': float(average_precision_score(y_val, val_pred)),
        'n_train_pairs': int(len(X_train)),
        'n_val_pairs': int(len(X_val)),
        'n_features': int(X.shape[1])
    }

    return model, feature_cols, metrics


def predict_pairs(cat_model, pairs_feat_df: pd.DataFrame, feature_cols: list):
    X = pairs_feat_df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(-999.0)
    pred = cat_model.predict_proba(X)[:, 1]
    out = pairs_feat_df[['profile_id_1', 'profile_id_2']].copy()
    if 'label' in pairs_feat_df.columns:
        out['label'] = pairs_feat_df['label']
    out['score'] = pred
    return out


# =========================================================
# 7. Saving / Loading
# =========================================================

def save_artifacts(
    output_dir: str,
    cat_model,
    feature_cols: list,
    schema: dict,
    gnn_model,
    train_metrics: dict,
    train_profile_df: pd.DataFrame,
    val_profile_df: pd.DataFrame = None,
    test_profile_df: pd.DataFrame = None,
    train_pairs_scored: pd.DataFrame = None,
    val_pairs_scored: pd.DataFrame = None,
    test_pairs_scored: pd.DataFrame = None,
):
    ensure_dir(output_dir)

    # CatBoost model
    cat_model_path = os.path.join(output_dir, 'catboost_pair_model.cbm')
    cat_model.save_model(cat_model_path)

    # Feature columns
    with open(os.path.join(output_dir, 'feature_cols.json'), 'w', encoding='utf-8') as f:
        json.dump(feature_cols, f, ensure_ascii=False, indent=2)

    # Schema
    with open(os.path.join(output_dir, 'graph_schema.pkl'), 'wb') as f:
        pickle.dump(schema, f)

    # GNN state dict
    torch.save(gnn_model.state_dict(), os.path.join(output_dir, 'graphsage_state_dict.pt'))

    # Metrics
    with open(os.path.join(output_dir, 'train_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump(train_metrics, f, ensure_ascii=False, indent=2)

    # Embeddings / profile tables
    train_profile_df.to_parquet(os.path.join(output_dir, 'train_profile_embeddings.parquet'), index=False)
    if val_profile_df is not None:
        val_profile_df.to_parquet(os.path.join(output_dir, 'validation_profile_embeddings.parquet'), index=False)
    if test_profile_df is not None:
        test_profile_df.to_parquet(os.path.join(output_dir, 'test_profile_embeddings.parquet'), index=False)

    # Pair scores
    if train_pairs_scored is not None:
        train_pairs_scored.to_parquet(os.path.join(output_dir, 'train_pair_scores.parquet'), index=False)
    if val_pairs_scored is not None:
        val_pairs_scored.to_parquet(os.path.join(output_dir, 'validation_pair_scores.parquet'), index=False)
    if test_pairs_scored is not None:
        test_pairs_scored.to_parquet(os.path.join(output_dir, 'test_pair_scores.parquet'), index=False)


# =========================================================
# 8. Train pipeline
# =========================================================

def run_train_pipeline_v2(
    df: pd.DataFrame,
    attribute_cols=None,
    max_freq_per_value=5000,
    max_group_size=200,
    knn_neighbors=20,
    neg_to_pos_ratio=5,
    hidden_channels=64,
    out_channels=64,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    seed=42
):
    set_seed(seed)

    print('1) Build train profile_df...')
    train_profile_df = build_profile_df(df, split='train')
    print('train_profile_df:', train_profile_df.shape)

    print('2) Fit graph schema on train...')
    schema = fit_graph_schema(
        train_profile_df,
        attribute_cols=attribute_cols,
        max_freq_per_value=max_freq_per_value
    )

    print('3) Build train heterograph...')
    train_data, train_meta = build_hetero_graph_from_schema(train_profile_df, schema)
    print(train_data)

    print('4) Train GraphSAGE...')
    gnn_model = train_graphsage(
        data=train_data,
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay
    )

    print('5) Infer train embeddings...')
    train_emb = infer_profile_embeddings(gnn_model, train_data)
    train_profile_df = attach_embeddings(train_profile_df, train_emb)

    print('6) Generate train candidates...')
    cand_2hop = generate_candidate_pairs_2hop(
        train_profile_df,
        attribute_cols=schema['attribute_cols'],
        max_group_size=max_group_size
    )
    cand_knn = generate_candidate_pairs_knn(
        train_profile_df,
        emb_prefix='emb_',
        n_neighbors=knn_neighbors
    )
    train_candidate_pairs = cand_2hop | cand_knn
    print('train candidates:', len(train_candidate_pairs))

    print('7) Label and sample train pairs...')
    train_pairs_all = label_candidate_pairs(train_candidate_pairs, train_profile_df)
    train_pairs_sampled = sample_pairs_for_training(
        train_pairs_all,
        neg_to_pos_ratio=neg_to_pos_ratio,
        random_state=seed
    )
    print(train_pairs_sampled['label'].value_counts())

    print('8) Build train pair features...')
    train_pairs_feat = build_pair_features(train_pairs_sampled, train_profile_df)

    print('9) Train CatBoost...')
    cat_model, feature_cols, train_metrics = train_catboost_pair_classifier(
        train_pairs_feat,
        random_state=seed
    )
    print('train metrics:', train_metrics)

    train_pairs_scored = predict_pairs(cat_model, train_pairs_feat, feature_cols)

    artifacts = {
        'schema': schema,
        'gnn_model': gnn_model,
        'cat_model': cat_model,
        'feature_cols': feature_cols,
        'train_metrics': train_metrics,
        'train_profile_df': train_profile_df,
        'train_pairs_scored': train_pairs_scored,
        'train_pairs_feat': train_pairs_feat,
    }
    return artifacts


# =========================================================
# 9. Inference pipeline for validation/test
# =========================================================

def run_split_inference(
    df: pd.DataFrame,
    split: str,
    schema: dict,
    gnn_model,
    cat_model,
    feature_cols: list,
    max_group_size=200,
    knn_neighbors=20
):
    print(f'Inference split={split}')

    profile_df = build_profile_df(df, split=split)
    print(f'{split}_profile_df:', profile_df.shape)

    data, meta = build_hetero_graph_from_schema(profile_df, schema)
    profile_emb = infer_profile_embeddings(gnn_model, data)
    profile_df = attach_embeddings(profile_df, profile_emb)

    cand_2hop = generate_candidate_pairs_2hop(
        profile_df,
        attribute_cols=schema['attribute_cols'],
        max_group_size=max_group_size
    )
    cand_knn = generate_candidate_pairs_knn(
        profile_df,
        emb_prefix='emb_',
        n_neighbors=knn_neighbors
    )
    candidate_pairs = cand_2hop | cand_knn
    print(f'{split} candidates:', len(candidate_pairs))

    pairs_df = label_candidate_pairs(candidate_pairs, profile_df)
    print(pairs_df['label'].value_counts(dropna=False))

    pairs_feat_df = build_pair_features(pairs_df, profile_df)
    scored_pairs = predict_pairs(cat_model, pairs_feat_df, feature_cols)

    if 'label' in scored_pairs.columns and scored_pairs['label'].nunique() > 1:
        metrics = {
            'roc_auc': float(roc_auc_score(scored_pairs['label'], scored_pairs['score'])),
            'pr_auc': float(average_precision_score(scored_pairs['label'], scored_pairs['score'])),
            'n_pairs': int(len(scored_pairs)),
        }
    else:
        metrics = {
            'roc_auc': None,
            'pr_auc': None,
            'n_pairs': int(len(scored_pairs)),
        }

    # === Дополнительная проверка: sampled universe (no blocking bias) ===
    # строим честный тест/валидационный сэмпл по entity_id

    # 1) профили и сущности для этого сплита
    split_df = df[df['split'] == split].copy()
    profile_to_entity = dict(zip(split_df['profile_id'], split_df['entity_id']))

    # 2) True pairs (upper bound) внутри entity_id
    from itertools import combinations
    true_pairs = []
    for eid, grp in split_df.groupby('entity_id'):
        profs = grp['profile_id'].tolist()
        if len(profs) >= 2:
            for a, b in combinations(profs, 2):
                true_pairs.append((min(a, b), max(a, b)))
    true_df = pd.DataFrame(true_pairs, columns=['profile_id_1', 'profile_id_2'])
    true_df['label'] = 1

    # 3) Негативы (пары из разных entity_id)
    all_profiles = split_df['profile_id'].tolist()
    rng = np.random.default_rng(42)
    neg_pairs = set()
    target_neg = min(len(true_df) * 10, 200_000)

    while len(neg_pairs) < target_neg:
        a, b = rng.choice(all_profiles, size=2, replace=False)
        if profile_to_entity[a] == profile_to_entity[b]:
            continue
        pair = (min(a, b), max(a, b))
        neg_pairs.add(pair)

    neg_df = pd.DataFrame(list(neg_pairs), columns=['profile_id_1', 'profile_id_2'])
    neg_df['label'] = 0

    sampled_pairs_df = pd.concat([true_df, neg_df], ignore_index=True)

    # 4) Фичи для sampled universe
    sampled_pairs_feat = build_pair_features(sampled_pairs_df, profile_df)

    # 5) Скоринг model_cb на тех же feature_cols
    sampled_scored = predict_pairs(cat_model, sampled_pairs_feat, feature_cols)

    if sampled_scored['label'].nunique() > 1:
        sampled_pr_auc = float(average_precision_score(
            sampled_scored['label'], sampled_scored['score']
        ))
        sampled_roc_auc = float(roc_auc_score(
            sampled_scored['label'], sampled_scored['score']
        ))
    else:
        sampled_pr_auc = None
        sampled_roc_auc = None

    sampled_metrics = {
        'roc_auc': sampled_roc_auc,
        'pr_auc': sampled_pr_auc,
        'true_pairs': int(len(true_df)),
        'neg_pairs': int(len(neg_df)),
    }

    print(f'{split} sampled-universe metrics:', sampled_metrics)



    return {
        'profile_df': profile_df,
        'pairs_df': pairs_df,
        'pairs_feat_df': pairs_feat_df,
        'scored_pairs': scored_pairs,
        'metrics': metrics,                  # candidate-universe
        'sampled_metrics': sampled_metrics,  # твоя честная проверка
    }


# =========================================================
# 10. Full train + inference + save
# =========================================================

def run_full_pipeline_v2(
    df: pd.DataFrame,
    output_dir: str = 'output/identity_graph_pipeline',
    attribute_cols=None,
    max_freq_per_value=5000,
    max_group_size=200,
    knn_neighbors=20,
    neg_to_pos_ratio=5,
    hidden_channels=64,
    out_channels=64,
    epochs=100,
    lr=1e-3,
    weight_decay=1e-5,
    seed=42,
    run_validation=True,
    run_test=True
):
    ensure_dir(output_dir)

    train_artifacts = run_train_pipeline_v2(
        df=df,
        attribute_cols=attribute_cols,
        max_freq_per_value=max_freq_per_value,
        max_group_size=max_group_size,
        knn_neighbors=knn_neighbors,
        neg_to_pos_ratio=neg_to_pos_ratio,
        hidden_channels=hidden_channels,
        out_channels=out_channels,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        seed=seed
    )

    val_artifacts = None
    test_artifacts = None

    if run_validation:
        val_artifacts = run_split_inference(
            df=df,
            split='validation',
            schema=train_artifacts['schema'],
            gnn_model=train_artifacts['gnn_model'],
            cat_model=train_artifacts['cat_model'],
            feature_cols=train_artifacts['feature_cols'],
            max_group_size=max_group_size,
            knn_neighbors=knn_neighbors
        )
        print('validation metrics:', val_artifacts['metrics'])

    if run_test:
        test_artifacts = run_split_inference(
            df=df,
            split='test',
            schema=train_artifacts['schema'],
            gnn_model=train_artifacts['gnn_model'],
            cat_model=train_artifacts['cat_model'],
            feature_cols=train_artifacts['feature_cols'],
            max_group_size=max_group_size,
            knn_neighbors=knn_neighbors
        )
        print('test metrics:', test_artifacts['metrics'])

    summary = {
        'train_metrics': train_artifacts['train_metrics'],
        'validation_metrics': val_artifacts['metrics'] if val_artifacts is not None else None,
        'test_metrics': test_artifacts['metrics'] if test_artifacts is not None else None,
        'config': {
            'attribute_cols': attribute_cols,
            'max_freq_per_value': max_freq_per_value,
            'max_group_size': max_group_size,
            'knn_neighbors': knn_neighbors,
            'neg_to_pos_ratio': neg_to_pos_ratio,
            'hidden_channels': hidden_channels,
            'out_channels': out_channels,
            'epochs': epochs,
            'lr': lr,
            'weight_decay': weight_decay,
            'seed': seed,
        }
    }

    with open(os.path.join(output_dir, 'summary.json'), 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    save_artifacts(
        output_dir=output_dir,
        cat_model=train_artifacts['cat_model'],
        feature_cols=train_artifacts['feature_cols'],
        schema=train_artifacts['schema'],
        gnn_model=train_artifacts['gnn_model'],
        train_metrics=train_artifacts['train_metrics'],
        train_profile_df=train_artifacts['train_profile_df'],
        val_profile_df=val_artifacts['profile_df'] if val_artifacts is not None else None,
        test_profile_df=test_artifacts['profile_df'] if test_artifacts is not None else None,
        train_pairs_scored=train_artifacts['train_pairs_scored'],
        val_pairs_scored=val_artifacts['scored_pairs'] if val_artifacts is not None else None,
        test_pairs_scored=test_artifacts['scored_pairs'] if test_artifacts is not None else None,
    )

    return {
        'train': train_artifacts,
        'validation': val_artifacts,
        'test': test_artifacts,
        'summary': summary,
    }

In [25]:
attribute_cols = {
    'domain': 'domain_set',
    'phone': 'phone_set',
    'device': 'device_set',
    'city': 'city_set',
    'region': 'region_set',
}

results = run_full_pipeline_v2(
    df=df,
    output_dir=ARTIFACTS_DIR,
    attribute_cols=attribute_cols,
    max_freq_per_value=3000,
    max_group_size=100,
    knn_neighbors=15,
    neg_to_pos_ratio=5,
    hidden_channels=64,
    out_channels=32,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-5,
    seed=42,
    run_validation=True,
    run_test=True
)

print(results['summary'])




1) Build train profile_df...
train_profile_df: (37069, 28)
2) Fit graph schema on train...
3) Build train heterograph...
HeteroData(
  profile={ x=[37069, 17] },
  domain={ x=[514, 2] },
  phone={ x=[2714, 2] },
  device={ x=[2, 2] },
  city={ x=[1357, 2] },
  region={ x=[305, 2] },
  (profile, has_domain, domain)={ edge_index=[2, 4695] },
  (domain, rev_has_domain, profile)={ edge_index=[2, 4695] },
  (profile, has_phone, phone)={ edge_index=[2, 2757] },
  (phone, rev_has_phone, profile)={ edge_index=[2, 2757] },
  (profile, has_device, device)={ edge_index=[2, 2931] },
  (device, rev_has_device, profile)={ edge_index=[2, 2931] },
  (profile, has_city, city)={ edge_index=[2, 29911] },
  (city, rev_has_city, profile)={ edge_index=[2, 29911] },
  (profile, has_region, region)={ edge_index=[2, 33385] },
  (region, rev_has_region, profile)={ edge_index=[2, 33385] }
)
4) Train GraphSAGE...
Epoch 010 | loss=17.0446
Epoch 020 | loss=8.1746
Epoch 030 | loss=4.6273
Epoch 040 | loss=2.9468
Epoc

/tmp/ipykernel_47744/1427381566.py:540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_min'] = pairs_df[[c1, c2]].min(axis=1)
/tmp/ipykernel_47744/1427381566.py:541: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_max'] = pairs_df[[c1, c2]].max(axis=1)
/tmp/ipykernel_47744/1427381566.py:539: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

9) Train CatBoost...
0:	test: 0.9698911	best: 0.9698911 (0)	total: 12.5ms	remaining: 6.26s
100:	test: 0.9940369	best: 0.9940369 (100)	total: 1.05s	remaining: 4.13s
200:	test: 0.9949004	best: 0.9949004 (200)	total: 2.08s	remaining: 3.1s
300:	test: 0.9954306	best: 0.9954308 (299)	total: 3.08s	remaining: 2.03s
400:	test: 0.9956709	best: 0.9956725 (399)	total: 4.16s	remaining: 1.03s
499:	test: 0.9957538	best: 0.9957828 (466)	total: 5.12s	remaining: 0us

bestTest = 0.9957827956
bestIteration = 466

Shrink model to first 467 iterations.
train metrics: {'roc_auc': 0.9957827956409653, 'pr_auc': 0.9862481098364723, 'n_train_pairs': 55785, 'n_val_pairs': 13947, 'n_features': 240}
Inference split=validation
validation_profile_df: (12474, 28)
validation candidates: 240010
label
0    235821
1      4189
Name: count, dtype: int64


/tmp/ipykernel_47744/1427381566.py:540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_min'] = pairs_df[[c1, c2]].min(axis=1)
/tmp/ipykernel_47744/1427381566.py:541: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_max'] = pairs_df[[c1, c2]].max(axis=1)
/tmp/ipykernel_47744/1427381566.py:539: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

validation sampled-universe metrics: {'roc_auc': 0.9960175799295308, 'pr_auc': 0.9826803377958644, 'true_pairs': 12399, 'neg_pairs': 123990}
validation metrics: {'roc_auc': 0.9953768783456943, 'pr_auc': 0.9404730509779639, 'n_pairs': 240010}
Inference split=test
test_profile_df: (12384, 28)
test candidates: 266460
label
0    262830
1      3630
Name: count, dtype: int64


/tmp/ipykernel_47744/1427381566.py:540: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_min'] = pairs_df[[c1, c2]].min(axis=1)
/tmp/ipykernel_47744/1427381566.py:541: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pairs_df[f'{col}_max'] = pairs_df[[c1, c2]].max(axis=1)
/tmp/ipykernel_47744/1427381566.py:539: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) in

test sampled-universe metrics: {'roc_auc': 0.9921573740415977, 'pr_auc': 0.9685610150691564, 'true_pairs': 6431, 'neg_pairs': 64310}
test metrics: {'roc_auc': 0.9934141080833551, 'pr_auc': 0.9146533054394309, 'n_pairs': 266460}
{'train_metrics': {'roc_auc': 0.9957827956409653, 'pr_auc': 0.9862481098364723, 'n_train_pairs': 55785, 'n_val_pairs': 13947, 'n_features': 240}, 'validation_metrics': {'roc_auc': 0.9953768783456943, 'pr_auc': 0.9404730509779639, 'n_pairs': 240010}, 'test_metrics': {'roc_auc': 0.9934141080833551, 'pr_auc': 0.9146533054394309, 'n_pairs': 266460}, 'config': {'attribute_cols': {'domain': 'domain_set', 'phone': 'phone_set', 'device': 'device_set', 'city': 'city_set', 'region': 'region_set'}, 'max_freq_per_value': 3000, 'max_group_size': 100, 'knn_neighbors': 15, 'neg_to_pos_ratio': 5, 'hidden_channels': 64, 'out_channels': 32, 'epochs': 50, 'lr': 0.001, 'weight_decay': 1e-05, 'seed': 42}}
